### Uvoz biblioteka

In [22]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# Podesavanje prikaza
print("Biblioteke su uspjesno ucitane!")

Biblioteke su uspjesno ucitane!


### Definisanje klasa 

In [23]:
CLASSES = {
    "Hydrolase": "hydrolase.tsv",
    "Transport protein": "transport_protein.tsv",
    "Transcription factor": "transcription_factor.tsv",
    "Receptor": "receptor.tsv",
    "Structural protein": "structural_protein.tsv"
}

### Učitavanje i spajanje TSV fajlova

In [24]:
all_dfs = []
for label, filename in CLASSES.items():
    df = pd.read_csv(f"../data/{filename}", sep="\t")
    df["label"] = label
    print(f"{label}: {len(df)} proteina")
    all_dfs.append(df)

dataset = pd.concat(all_dfs, ignore_index=True)
print(f"\nUkupno učitano proteina: {len(dataset)}")

Hydrolase: 2407 proteina


Transport protein: 1242 proteina
Transcription factor: 1418 proteina
Receptor: 1605 proteina
Structural protein: 774 proteina

Ukupno učitano proteina: 7446


### Upoznavanje sa skupom podataka

In [25]:
# Dimenzije i kolone
print(f"Dimenzije dataseta: {dataset.shape}")
print(f"Broj klasa: {dataset['label'].nunique()}")
print(f"Kolone: {list(dataset.columns)}\n")

Dimenzije dataseta: (7446, 5)
Broj klasa: 5
Kolone: ['Entry', 'Sequence', 'Protein names', 'Keywords', 'label']



In [26]:
# Prikaz prvih redova
print("Prvih 5 redova:")
dataset.head()

Prvih 5 redova:


,Entry,Sequence,Protein names,Keywords,label
0,A0A1B0GTW7,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,Ciliated left-right organizer metallopeptidase...,Alternative splicing;Disease variant;Glycoprot...,Hydrolase
1,A1A4Y4,MEAMNVEKASADGNLPEVISNIKETLKIVSRTPVNITMAGDSGNGM...,Immunity-related GTPase family M protein (EC 3...,Alternative splicing;Autophagy;Cell membrane;C...,Hydrolase
2,A1KZ92,MEPRLFCWTTLFLLAGWCLPGLPCPSRCLCFKSTVRCMHLMLDHIP...,Probable oxidoreductase PXDNL (EC 1.-.-.-) (Ca...,Alternative splicing;Calcium;Cell membrane;Cyt...,Hydrolase
3,A1Z1Q3,MYPSNKKKKVWREEKERLLKMTLEERRKEYLRDYIPLNSILSWKEE...,ADP-ribose glycohydrolase MACROD2 (MACRO domai...,3D-structure;Alternative splicing;DNA damage;H...,Hydrolase
4,A2A288,MEHPSKMEFFQKLGYDREDVLRVLGKLGEGALVNDVLQELIRTGSR...,Probable ribonuclease ZC3H12D (EC 3.1.-.-) (MC...,Alternative splicing;Chromosomal rearrangement...,Hydrolase


In [27]:
# Provjera nedostajucih vrijednosti
print("Broj nedostajucih vrijednosti po kolonama:")
print(dataset.isnull().sum())

Broj nedostajucih vrijednosti po kolonama:
Entry            0
Sequence         0
Protein names    0
Keywords         0
label            0
dtype: int64


In [28]:
# Identifikovanje dupliranih sekvenci i multifunkcionalnih proteina
dup_count = dataset['Sequence'].duplicated(keep=False).sum()
unique_seq_count = dataset['Sequence'].nunique()

print(f"Ukupno unikatnih sekvenci: {unique_seq_count}")
print(f"Broj redova sa dupliranim sekvencama: {dup_count}")

Ukupno unikatnih sekvenci: 7044
Broj redova sa dupliranim sekvencama: 788


### Pretprocesiranje i čišćenje sekvenici

Uklanjamo prazne redove, sekvence kraće od 10 aminokiselina i sekvence koje sadrže nestandardne aminokiseline (B, J, O, U, X, Z).

In [29]:
# Redovi sa praznom sekvencom
print(f"Broj redova sa praznom sekvencom: {dataset['Sequence'].isna().sum()}")

Broj redova sa praznom sekvencom: 0


In [30]:
def valid_sequence(seq):
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    seq = str(seq).upper().strip()

    unvalid = [aa for aa in seq if aa not in valid_aa]
    if unvalid:
        print(f"Nevalidni karakteri: {set(unvalid)}")
        return None
    
    if len(seq) < 10:
        print(f"Prekratka sekvenca: {len(seq)}")
        return None
    
    return seq

# Validacija
dataset['Sequence'] = dataset['Sequence'].apply(valid_sequence)

total_before = len(dataset)
dataset = dataset.dropna(subset=['Sequence'])
total_after = len(dataset)

print(f"Prije odbacivanja: {total_before} proteina")
print(f"Nakon odbacivanja: {total_after} proteina")
print(f"Uklonjeno nevalidnih sekvenci: {total_before - total_after}")

Nevalidni karakteri: {'U'}
Prije odbacivanja: 7446 proteina
Nakon odbacivanja: 7445 proteina
Uklonjeno nevalidnih sekvenci: 1


### Grupisanje dataseta po sekvenci

In [31]:
grouped = dataset.groupby('Sequence').agg({
    'Entry': 'first',
    'label': lambda x: sorted(set(x))
}).reset_index()

grouped['num_classes'] = grouped['label'].apply(len)
print(f"Ukupno unikatnih proteinskih sekvenci: {len(grouped)}")
print(f"\nDistribucija broja klasa po sekvenci:")
print(grouped['num_classes'].value_counts().sort_index())

Ukupno unikatnih proteinskih sekvenci: 7043

Distribucija broja klasa po sekvenci:
num_classes
1    6663
2     371
3       9
Name: count, dtype: int64


### Formiranje Single-Class (SC) i Multi-Class (MC) skupova

Grupišemo klase po unikatnim sekvencama i dijelimo podatke na dva skupa:
- **SC (Single-Class):** Sekvence sa tačno 1 funkcionalnom klasom
- **MC (Multi-Class):** Sekvence sa 2 ili više funkcionalnih klasa

In [32]:
grouped['is_multilabel'] = grouped['num_classes'] > 1

sc_count = (~grouped['is_multilabel']).sum()
mc_count = grouped['is_multilabel'].sum()

print(f"Single-Class (SC) proteina: {sc_count}")
print(f"Multi-Class (MC) proteina:  {mc_count}")

Single-Class (SC) proteina: 6663
Multi-Class (MC) proteina:  380


### Multi-hot enkodiranje klasa

In [36]:
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(grouped['label'])
label_df = pd.DataFrame(label_matrix, columns=mlb.classes_)

print(f"Klase (redoslijed kolona u multi-hot matrici): {list(mlb.classes_)}")

dataset_final = pd.concat([grouped.reset_index(drop=True), label_df], axis=1)
dataset_final.head()

Klase (redoslijed kolona u multi-hot matrici): ['Hydrolase', 'Receptor', 'Structural protein', 'Transcription factor', 'Transport protein']


,Sequence,Entry,label,num_classes,is_multilabel,Hydrolase,Receptor,Structural protein,Transcription factor,Transport protein
0,DKQLDADVSPKPTIFLPSIAETKLQKAGTYLCLLEKFFPDIIKIHW...,P03986,[Receptor],1,False,0,1,0,0,0
1,MAAAAAAAAAVGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...,Q9NRU3,[Transport protein],1,False,0,0,0,0,1
2,MAAAAAALSGAGTPPAGGGAGGGGAGGGGSPPGGWAVARLEGREFE...,Q01167,[Transcription factor],1,False,0,0,0,1,0
3,MAAAAAEEGMEPRALQYEQTLMYGRYTQDLGAFAKEEAARIRLGGP...,P51788,[Transport protein],1,False,0,0,0,0,1
4,MAAAAALRAPAQSSVTFEDVAVNFSLEEWSLLNEAQRCLYRDVMLE...,Q9NXT0,[Transcription factor],1,False,0,0,0,1,0


In [34]:
# Svaki protein sa >1 labelom doprinosi u više kolona istovremeno
print("Broj pojavljivanja po klasi (uzimajući u obzir multifunkcionalne proteine):")
print(label_df.sum().sort_values(ascending=False))

Broj pojavljivanja po klasi (uzimajući u obzir multifunkcionalne proteine):
Hydrolase               2403
Receptor                1602
Transcription factor    1414
Transport protein       1241
Structural protein       772
dtype: int64


### Čuvanje obrađenih podataka

In [ ]:
processed_dir = os.path.join("..", "data", "processed")
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir, exist_ok=True)

# Glavni dataset za dalji rad
final_path = os.path.join(processed_dir, "dataset_final.csv")
dataset_final.to_csv(final_path, index=False)

# Sporedni fajlovi - SC vs. MC
sc_path = os.path.join(processed_dir, "df_sc_reference.csv")
mc_path = os.path.join(processed_dir, "df_mc_reference.csv")

df_sc_ref = dataset_final[~dataset_final['is_multilabel']].copy()
df_mc_ref = dataset_final[dataset_final['is_multilabel']].copy()

df_sc_ref.to_csv(sc_path, index=False)
df_mc_ref.to_csv(mc_path, index=False)

print("Podaci su uspješno sačuvani u 'data/processed/':")
print(f" - {final_path}  (glavni dataset, {len(dataset_final)} proteina)")
print(f" - {sc_path}  (referentni SC skup, {len(df_sc_ref)} proteina)")
print(f" - {mc_path}  (referentni MC skup, {len(df_mc_ref)} proteina)")

Podaci su uspješno sačuvani u 'data/processed/':
 - ..\data\processed\dataset_final.csv  (glavni dataset, 7043 proteina)
 - ..\data\processed\df_sc_reference.csv  (referentni SC skup, 6663 proteina)
 - ..\data\processed\df_mc_reference.csv  (referentni MC skup, 380 proteina)
